In [2]:
import os, time, math
from collections import Counter
import numpy as np
import pandas as pd
from datasets import load_dataset
from openai import OpenAI
from tqdm.auto import tqdm

# ==========================================
# 1. CONFIGURATION FOR LM STUDIO
# ==========================================
BASE_URL = "http://172.17.103.35:1234/v1"
API_KEY = "lm-studio" # Default placeholder for LM studio
MODEL_ID = "local-model" # LM Studio uses whichever model is currently loaded
N_QUESTIONS = 400  
OUTPUT_DIR = "results_final_offline"
CSV_PATH = f"{OUTPUT_DIR}/LLAMA_MED42_offline.csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Initialize the OpenAI client pointing to the local LM Studio server
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

# ==========================================
# 2. LOAD DATASET
# ==========================================
print("Loading MedMCQA dataset...")
ds = load_dataset("medmcqa", "default", split=f"validation[:{N_QUESTIONS}]")
df = pd.DataFrame(ds)

# Map options A-D and correct letter
df['A'] = df['opa']
df['B'] = df['opb']
df['C'] = df['opc']
df['D'] = df['opd']
df['correct_letter'] = df['cop'].map({0:'A', 1:'B', 2:'C', 3:'D'})

print(f"✅ Loaded {len(df)} questions.")

# ==========================================
# 3. HELPER FUNCTIONS
# ==========================================
def sample_answers(prompt, m=6, temp=0.8):
    """Sample m answers from the local LM Studio model."""
    answers = []
    for _ in tqdm(range(m), desc="Sampling answers", leave=False):
        try:
            resp = client.chat.completions.create(
                model=MODEL_ID,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=64,
                temperature=temp,
            )
            
            if resp.choices and resp.choices[0].message.content:
                answers.append(resp.choices[0].message.content.strip())
            else:
                answers.append("LETTER: X\nREASON: Empty response.")
                
        except Exception as e:
            print(f"⚠️ API Error: {e}. Retrying in 2s...")
            time.sleep(2) # Give the local server a moment to recover
            answers.append("LETTER: X\nREASON: API Error occurred.")
            
    return answers

def normalize_answer(question, options, raw_answer):
    """Map raw answer -> LETTER + canonical reason."""
    A, B, C, D = options
    norm_prompt = f"""Medical evaluator: map answer to exact option letter + short standardized reason.

Question: {question}

Options:
A. {A}
B. {B}
C. {C}
D. {D}

Model answer: {raw_answer}

Respond exactly:
LETTER: A/B/C/D/X
REASON: [1 sentence explanation]"""

    try:
        resp = client.chat.completions.create(
            model=MODEL_ID,
            messages=[{"role": "user", "content": norm_prompt}],
            max_tokens=80,
            temperature=0.0,
        )
        text = resp.choices[0].message.content.strip()
    except Exception as e:
        text = "LETTER: X\nREASON: API Error during normalization."
    
    letter = "X"
    reason = text
    for line in text.splitlines():
        if line.upper().startswith("LETTER:"):
            # Extract just the letter
            for c in line.upper().replace("LETTER:", "").strip():
                if c in "ABCDX":
                    letter = c
                    break
        if line.upper().startswith("REASON:"):
            reason = line[7:].strip()
    return letter, reason

def semantic_entropy(reasons):
    """Entropy over canonical reasons (same reason = same semantic cluster)."""
    counts = Counter(reasons)
    total = sum(counts.values())
    probs = [c/total for c in counts.values()]
    return -sum(p*math.log(p+1e-12) for p in probs)


# ==========================================
# 4. RESUMABLE MAIN LOOP
# ==========================================
records = []
start_index = 0

# Auto-resume logic
if os.path.exists(CSV_PATH):
    print(f"\n📂 Found existing progress at {CSV_PATH}")
    try:
        existing_df = pd.read_csv(CSV_PATH)
        records = existing_df.to_dict('records')
        start_index = len(records)
        print(f"▶️ Resuming from question index {start_index} (Progress: {start_index}/{len(df)})...")
    except Exception as e:
        print(f"⚠️ Could not load existing CSV ({e}). Starting from 0.")

for i in tqdm(range(start_index, len(df)), desc="Overall Progress", initial=start_index, total=len(df)):
    row = df.iloc[i]
    q = row['question']
    options = (row['A'], row['B'], row['C'], row['D'])
    correct = row['correct_letter']
    
    prompt = f"""Medical MCQ. Choose A/B/C/D and explain briefly.

Question: {q}

A. {options[0]}
B. {options[1]}
C. {options[2]}
D. {options[3]}

Answer:"""

    raw_answers = sample_answers(prompt, m=6, temp=0.8)
    
    norm_letters = []
    norm_reasons = []
    
    for ans in tqdm(raw_answers, desc="Normalizing", leave=False):
        letter, reason = normalize_answer(q, options, ans)
        norm_letters.append(letter)
        norm_reasons.append(reason)
    
    H = semantic_entropy(norm_reasons)
    
    letter_counts = Counter(norm_letters)
    maj_letter, maj_count = letter_counts.most_common(1)[0]
    
    # Safely handle majority reason extraction
    maj_reasons = [r for l, r in zip(norm_letters, norm_reasons) if l == maj_letter]
    maj_reason = Counter(maj_reasons).most_common(1)[0][0] if maj_reasons else "N/A"
    
    is_correct = (maj_letter == correct)
    hallucinated = (maj_letter != "X" and maj_letter != correct)
    
    records.append({
        'question_id': i,
        'question': q,
        'option_A': options[0],
        'option_B': options[1],
        'option_C': options[2],
        'option_D': options[3],
        'correct_letter': correct,
        'majority_letter': maj_letter,
        'majority_reason': maj_reason,
        'semantic_entropy': H,
        'is_correct': is_correct,
        'hallucinated': hallucinated,
        'samples_count': len(raw_answers),
    })
    
    # Real-Time Metric Calculation
    acc = np.mean([r['is_correct'] for r in records])
    halluc_rate = np.mean([r['hallucinated'] for r in records])
    
    print(f"\n--- Progress: {i+1}/{len(df)} ---")
    print(f"Real-time Acc: {acc:.1%} | Real-time Hallucination: {halluc_rate:.1%}")
    
    # Save Every 2 Questions
    if (i + 1) % 2 == 0:
        pd.DataFrame(records).to_csv(CSV_PATH, index=False)
        print(f"💾 Saved intermediate results to {CSV_PATH}")
    
    # Brief pause to let local hardware breathe
    time.sleep(0.5) 

# ==========================================
# 5. FINAL RESULTS
# ==========================================
res_df = pd.DataFrame(records)
total = len(res_df)
correct_count = res_df['is_correct'].sum()
halluc = res_df['hallucinated'].sum()

print("\n🎯 MEDICAL QA HALLUCINATION BENCHMARK (LM STUDIO)")
print("=" * 50)
print(f"Accuracy: {(correct_count/total)*100:.1f}% ({correct_count}/{total})")
print(f"Hallucination rate: {(halluc/total)*100:.1f}% ({halluc}/{total})")

res_df.to_csv(CSV_PATH, index=False)
print("💾 Final Save:", CSV_PATH)

Loading MedMCQA dataset...


✅ Loaded 400 questions.

📂 Found existing progress at results_final_offline/LLAMA_MED42_offline.csv
▶️ Resuming from question index 74 (Progress: 74/400)...


Overall Progress:  18%|#8        | 74/400 [00:00<?, ?it/s]

Sampling answers:   0%|          | 0/6 [00:00<?, ?it/s]

Normalizing:   0%|          | 0/6 [00:00<?, ?it/s]

...

Normalizing:   0%|          | 0/6 [00:00<?, ?it/s]


--- Progress: 399/400 ---
Real-time Acc: 56.1% | Real-time Hallucination: 35.8%


Sampling answers:   0%|          | 0/6 [00:00<?, ?it/s]

Normalizing:   0%|          | 0/6 [00:00<?, ?it/s]


--- Progress: 400/400 ---
Real-time Acc: 56.0% | Real-time Hallucination: 36.0%
💾 Saved intermediate results to results_final_offline/LLAMA_MED42_offline.csv

🎯 MEDICAL QA HALLUCINATION BENCHMARK (LM STUDIO)
Accuracy: 56.0% (224/400)
Hallucination rate: 36.0% (144/400)
💾 Final Save: results_final_offline/LLAMA_MED42_offline.csv
